# Erdős–Straus Conjecture Solver — Google Colab

**Guinea Pig Trench LLC**

---

The [Erdős–Straus conjecture](https://en.wikipedia.org/wiki/Erd%C5%91s%E2%80%93Straus_conjecture) states that for every integer $n \geq 2$, the fraction $\frac{4}{n}$ can be written as a sum of three unit fractions:

$$\frac{4}{n} = \frac{1}{x} + \frac{1}{y} + \frac{1}{z}$$

where $x, y, z$ are positive integers with $x \leq y \leq z$.

This has been computationally verified up to $n = 10^{14}$. Our solver has independently verified it to $10^8$ and this notebook lets you:

- **Verify further ranges** beyond what we've already checked
- **Re-verify existing ranges** to confirm results
- **Upload a chunk file** of specific $n$ values to solve

Only $n \equiv 1$ or $17 \pmod{24}$ need brute-force search — all other residues have known parametric decompositions.

GitHub: [https://github.com/Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)

In [ ]:
# === Cell 2: Setup — detect runtime type and hardware ===
import os
import multiprocessing as mp

print("=" * 55)
print("  Runtime Detection")
print("=" * 55)

# Check for GPU
gpu_available = False
try:
    gpu_info = os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip()
    if gpu_info:
        gpu_available = True
        print(f"GPU detected: {gpu_info}")
    else:
        print("No GPU detected (CPU-only runtime)")
except Exception:
    print("No GPU detected (CPU-only runtime)")

# CPU info
cpu_count = mp.cpu_count()
print(f"CPU cores: {cpu_count}")

# Set workers: Colab free tier gives 2 CPUs, Pro gives more
# GPU runtimes still use CPU for this solver (pure integer math)
workers = max(1, cpu_count)
print(f"Workers: {workers}")
print()
print("Note: This solver is pure CPU (integer arithmetic).")
print("A GPU runtime still helps because Colab often gives")
print("more CPU cores and RAM to GPU instances.")
print("=" * 55)

In [ ]:
# === Cell 3: The Solver (self-contained, zero dependencies) ===
import math


def solve_single(n, step_cap=10_000_000, y_cap_per_x=1_000_000):
    """Find x <= y <= z such that 4/n = 1/x + 1/y + 1/z.

    Uses a per-x y-iteration cap to avoid exhausting the global budget on a
    single x value.  Falls back to the global step_cap as an overall limit.

    Returns dict with {n, x, y, z, steps} on success, or None.
    """
    if n <= 0:
        return None
    if n == 1:
        return None

    steps = 0
    x_min = max(1, math.ceil(n / 4))
    x_max = n

    for x in range(x_min, x_max + 1):
        num_r = 4 * x - n
        if num_r <= 0:
            steps += 1
            if steps >= step_cap:
                return None
            continue
        den_r = n * x

        y_min = math.ceil(den_r / num_r)
        y_max = 2 * den_r // num_r

        y_steps = 0
        for y in range(max(x, y_min), y_max + 1):
            steps += 1
            y_steps += 1
            if steps >= step_cap:
                return None
            if y_steps >= y_cap_per_x:
                break

            denom_z = num_r * y - den_r
            if denom_z <= 0:
                continue
            num_z = den_r * y
            if num_z % denom_z == 0:
                z = num_z // denom_z
                if z >= y:
                    return {"n": n, "x": x, "y": y, "z": z, "steps": steps}

    return None


def worker(args):
    """Multiprocessing worker. Returns a result dict for every n."""
    n, cap = args
    result = solve_single(n, step_cap=cap)
    if result:
        result["solved"] = True
        return result
    return {"n": n, "x": 0, "y": 0, "z": 0, "steps": cap, "solved": False}


print("Solver loaded.")
print("  solve_single(n, step_cap, y_cap_per_x) -> dict or None")
print("  worker((n, cap)) -> dict with 'solved' flag")
print()
# Quick sanity check
test = solve_single(5)
assert test is not None and test["x"] == 2 and test["y"] == 4 and test["z"] == 20, "Sanity check failed!"
print(f"Sanity check: 4/5 = 1/{test['x']} + 1/{test['y']} + 1/{test['z']}  ({test['steps']} steps)")

---

## Choose Your Mode

Run **one** of the next two cells:

- **Option A** — Generate a fresh range of hard-residue targets (n mod 24 in {1, 17})
- **Option B** — Upload a chunk file containing specific n values to solve

In [ ]:
# === Cell 5: Option A — Generate hard-residue targets for a custom range ===
#
# Set your range here. Only n where n%24 in {1, 17} need brute-force;
# all other residues have known parametric decompositions.

n_start = 100_000_001   # <-- edit this
n_end   = 100_100_000   # <-- edit this

hard_residues = {1, 17}
ns = [n for n in range(n_start, n_end + 1) if n % 24 in hard_residues]

print(f"Range: {n_start:,} to {n_end:,}")
print(f"Hard-residue targets (n%24 in {{1,17}}): {len(ns):,}")
print(f"Skipped (trivially solvable): {(n_end - n_start + 1) - len(ns):,}")

In [ ]:
# === Cell 6: Option B — Upload a chunk file ===
#
# The file should have one integer per line.
# (e.g., survivors from a previous run)

from google.colab import files

print("Upload your chunk file (one n per line):")
uploaded = files.upload()
chunk_file = list(uploaded.keys())[0]

with open(chunk_file) as f:
    ns = [int(line.strip()) for line in f if line.strip()]

print(f"\nLoaded {len(ns):,} values from {chunk_file}")
print(f"Range: {min(ns):,} to {max(ns):,}")

In [ ]:
# === Cell 7: Run the Solver ===
import multiprocessing as mp
import csv
import time

# --- Configuration ---
# step_cap: max iterations per n value.
#   10M (default) is fast and catches almost everything.
#   100M catches everything we've tested but is ~10x slower.
step_cap = 10_000_000

batch_size = workers * 8
total = len(ns)
total_batches = (total + batch_size - 1) // batch_size
solved_count = 0
all_results = []

out_file = "colab_results.csv"
fields = ["n", "x", "y", "z", "steps", "solved"]

print("=" * 55)
print(f"  Solving {total:,} values")
print(f"  step_cap = {step_cap:,}")
print(f"  workers  = {workers}")
print(f"  batches  = {total_batches:,} (batch_size={batch_size})")
print("=" * 55)
print()

t0 = time.time()

with open(out_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()

    with mp.Pool(workers) as pool:
        for i in range(0, total, batch_size):
            batch = ns[i : i + batch_size]
            batch_num = i // batch_size + 1
            bt = time.time()

            results = pool.map(worker, [(n, step_cap) for n in batch])

            elapsed = time.time() - bt
            batch_solved = 0

            for r in results:
                writer.writerow(r)
                all_results.append(r)
                if r["solved"]:
                    batch_solved += 1
                    solved_count += 1

            if batch_num % 25 == 0 or batch_num == total_batches:
                f.flush()
                pct = 100 * (i + len(batch)) / total
                rate = (i + len(batch)) / (time.time() - t0)
                print(
                    f"  [{pct:5.1f}%] batch {batch_num}/{total_batches}: "
                    f"{batch_solved}/{len(batch)} solved "
                    f"({elapsed:.1f}s) | total: {solved_count:,}/{i + len(batch):,} "
                    f"| {rate:.0f} n/s"
                )

total_time = time.time() - t0
print()
print("=" * 55)
print(f"  Done in {total_time:.1f}s")
print(f"  Solved: {solved_count:,} / {total:,}")
if total_time > 0:
    print(f"  Rate:   {total / total_time:.0f} n/s")
print(f"  Output: {out_file}")
print("=" * 55)

In [ ]:
# === Cell 8: Save results to Google Drive + download ===
import shutil
from google.colab import drive, files

# Mount Google Drive
drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/erdos_straus"
os.makedirs(drive_dir, exist_ok=True)

# Copy results to Drive (survives runtime disconnects)
drive_path = os.path.join(drive_dir, out_file)
shutil.copy2(out_file, drive_path)
print(f"Saved to Google Drive: {drive_path}")

# Also offer direct download
print("\nDownloading to your machine...")
files.download(out_file)

In [ ]:
# === Cell 9: Quick Stats ===

solved   = [r for r in all_results if r["solved"]]
unsolved = [r for r in all_results if not r["solved"]]

print("=" * 55)
print("  Results Summary")
print("=" * 55)
print(f"  Total:    {len(all_results):,}")
print(f"  Solved:   {len(solved):,}")
print(f"  Unsolved: {len(unsolved):,}")
print()

if solved:
    max_z = max(r["z"] for r in solved)
    max_z_row = [r for r in solved if r["z"] == max_z][0]
    print(f"  Largest z found: {max_z:,}")
    print(f"    n={max_z_row['n']}, x={max_z_row['x']}, y={max_z_row['y']}, z={max_z_row['z']}")
    print()

    avg_steps = sum(r["steps"] for r in solved) / len(solved)
    max_steps = max(r["steps"] for r in solved)
    print(f"  Avg steps (solved): {avg_steps:,.0f}")
    print(f"  Max steps (solved): {max_steps:,}")

if unsolved:
    print()
    print("  *** UNSOLVED values (potential counterexamples!): ***")
    for r in unsolved[:50]:
        print(f"    n = {r['n']:,}")
    if len(unsolved) > 50:
        print(f"    ... and {len(unsolved) - 50:,} more")
    print()
    print(f"  These likely need a higher step_cap (try 100_000_000).")
else:
    print()
    print("  All values solved! The conjecture holds for this range.")

print("=" * 55)